# Flight Fuel Analysis — reproducible walkthrough

**Research question:** When a fuel estimator serves only its lower-risk intervals, which durations and aircraft types disappear? What error does it cost to guarantee broader access?

This notebook is a short technical companion to the [offline dashboard](../demo/index.html). It **replays the real policy** on 7,619 committed September validation rows and checks its numbers against the published result. It does not train a model or claim a new untouched test. Run all cells from the repository root; the first cell also handles opening the notebook from `notebooks/`. Only the packages in `requirements.txt` are needed.

**Presentation route:** provenance → data → model → selection → tradeoff → failure cases → limitations.

## 1. Where the data come from

The [EUROCONTROL PRC 2025 Aircraft Fuel Burn Estimation Data Challenge](https://doi.org/10.5281/zenodo.19184662) publishes `flightlist_*.parquet` (aircraft/flight metadata), `fuel_*.parquet` (interval start/end and `fuel_kg` labels), and `flights_*.zip` (one trajectory Parquet per flight). The repository downloads them from `https://zenodo.org/api/records/19184662/files/<filename>/content` in `data_audit.py` and `download_archives.py`, verifies expected MD5 checksums, and writes raw files to ignored `data/raw/`.

`features.py` selects **ADS-B observations only** within the supplied half-open `[start, end)` interval, then computes counts, raw altitude/speed summaries and observation-gap features. It saves `data/processed/features_<split>.parquet` locally. The label is the difference between two reported fuel-on-board values, **not a direct fuel-flow reading**. Interval boundaries are supplied by the benchmark. Neither labels nor future trajectories are used to construct earlier interval features.

The expensive ZIPs are **not needed** to run this notebook. It uses the generated dashboard's committed validation rows (type, duration, risk score, predicted fuel, reference fuel, ADS-B count and deterministic tie), plus `reports/eligibility-results.json`.

In [ ]:
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "reports/eligibility-results.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "reports/eligibility-results.json").exists(), "Open from the repo root or notebooks/"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.flight_fuel.eligibility import duration_groups, select

results = json.loads((ROOT / "reports/eligibility-results.json").read_text())
html = (ROOT / "demo/index.html").read_text()
assert html.count("const DATA=") == 1, "Regenerate the dashboard from eligibility.py"
# The dashboard embeds ordinary JSON; parse it without executing browser JavaScript.
payload, _ = json.JSONDecoder().raw_decode(html.split("const DATA=", 1)[1])
columns = ["aircraft_type", "duration_min", "risk_score_kg", "prediction_kg",
           "reference_fuel_kg", "adsb_positions", "tie"]
frame = pd.DataFrame(payload["rows"], columns=columns)
frame["duration_group"] = duration_groups(frame.duration_min.to_numpy())
frame["absolute_error_kg"] = (frame.prediction_kg - frame.reference_fuel_kg).abs()
assert len(frame) == results["periods"]["validation"]["intervals"] == 7619
print(f"{len(frame):,} completed September intervals; {frame.aircraft_type.nunique()} aircraft types")
print("Published analysis:", results["status"])

## 2. The evaluation timeline and inputs

| Period | Role | Fits a model? |
| --- | --- | --- |
| April–August 2025 | Fit fuel estimator and type-rate fallback | Yes |
| September 1–9 | Fit model that ranks predicted absolute error | Yes |
| September 10–18 | Calibration block in historical protocol | No new fit here |
| September 19–27 | This notebook's labeled batch | No |
| October 2025 | Exploratory comparison | No |

Flights crossing September block boundaries are excluded. Flights do not overlap across the train/rank/final published splits. Routes can overlap. This is a **retrospective** study: we know the ACARS-defined start and end of a completed interval. The validation labels are used **after** predictions and policy selection to measure errors.

In [ ]:
display(pd.DataFrame({
    "value": [len(frame), frame.aircraft_type.nunique(), int((frame.duration_min < 5).sum()),
              int((frame.adsb_positions == 0).sum()), frame.reference_fuel_kg.median()],
    "meaning": ["validation intervals", "aircraft types", "under 5 minutes",
                "no usable ADS-B positions", "median reference fuel (kg)"]
}).set_index("meaning"))

counts = frame.duration_group.value_counts().reindex(["<5 min", "5–15 min", "15–60 min", "60+ min"]).fillna(0)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(counts.index, counts.values, color="#267c78")
ax.set(title="September validation is dominated by short intervals",
       ylabel="Number of intervals", xlabel="Supplied interval duration")
for i, n in enumerate(counts):
    ax.text(i, n + 50, f"{int(n):,}", ha="center", fontsize=9)
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()

**How to explain this plot:** The selection problem concerns *intervals*, not full-flight totals. Many labels describe less than five minutes, even though the publisher README describes a five-minute lower bound. We retain and disclose them. Zero ADS-B intervals use a training-only aircraft-type median rate times duration; they are not filled using the validation fuel label.

## 3. How accurate is the fuel model before selection?

For intervals with ADS-B observations, a histogram gradient boosting regressor uses aircraft type, duration and trajectory summaries. A separate model predicts an **absolute-error ranking score** from early-September residuals. The risk score is a ranking aid, **not** a probability or calibrated confidence interval. The revised fuel fit disables the historical row-random internal early-stopping split.

The table and plots below describe **all** validation intervals, including cases a selector may later reject. Absolute error is `|predicted kg − reference kg|`; MAE is its mean. RMSE penalizes larger misses more strongly. R² depends on the spread of fuel values and does not guarantee accurate individual estimates.

In [ ]:
d = results["periods"]["validation"]["residual_diagnostics"]["full"]
summary = pd.Series({
    "MAE (kg)": d["mae_kg"], "RMSE (kg)": d["rmse_kg"],
    "Median absolute error (kg)": d["median_absolute_error_kg"],
    "95th percentile absolute error (kg)": d["p95_absolute_error_kg"],
    "R²": d["r2"], "WAPE (%)": 100 * d["wape"]
}).round(2)
assert np.isclose(frame.absolute_error_kg.mean(), d["mae_kg"])
display(summary.to_frame("September, full batch"))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(frame.absolute_error_kg.clip(upper=1000), bins=40, color="#267c78")
axes[0].axvline(d["p95_absolute_error_kg"], color="#b65b32", linestyle="--", label="95th percentile")
axes[0].set(xlabel="Absolute error (kg; values >1,000 shown at 1,000)", ylabel="Intervals",
            title="Typical and tail errors")
axes[0].legend()
by_duration = results["periods"]["validation"]["residual_diagnostics"]["by_duration"]
order = [name for name in ["<5 min", "5–15 min", "15–60 min", "60+ min"] if name in by_duration]
axes[1].bar(order, [by_duration[name]["mae_kg"] for name in order], color="#b65b32")
axes[1].set(xlabel="Interval duration", ylabel="MAE on all intervals (kg)",
            title="Errors differ by duration")
for ax in axes: ax.tick_params(axis="x", rotation=12)
plt.tight_layout()
plt.show()

**Viva note:** The histogram clips the *display* at 1,000 kg; metrics use every original error. The 95th percentile is 372.75 kg, so a single headline R² of 0.872 hides substantial misses. The percentage error per individual interval (MAPE) is fragile when the reference fuel is near zero; WAPE is reported for this particular sample instead.

## 4. Replay the two selection rules — without using labels

Both policies start from the **same predictions and risk scores**. At 75% overall acceptance, `risk_only` selects the lowest scores. `representation_floor` first serves the lowest-risk rows within each duration bin and each aircraft type having at least 50 rows until its 50% access floor is met, then fills the overall target. The rules use type, duration, risk score and stable tie values; **reference fuel is deliberately absent from the selector call**.

We verify counts and metrics against the committed result JSON. The `accepted_reference_fuel_fraction` below is computed with labels **only after** selection and cannot be known on a new unlabeled batch.

In [ ]:
policies = results["periods"]["validation"]["policies"]
selected = {}
for name, floor in [("risk_only", 0.0), ("representation_floor", 0.5)]:
    mask, details = select(frame.risk_score_kg.to_numpy(),
                           frame.aircraft_type.to_numpy(),
                           frame.duration_min.to_numpy(),
                           frame.tie.to_numpy(),
                           target=0.75, floor=floor, min_type_n=50)
    selected[name] = mask
    published = policies[name]
    assert int(mask.sum()) == published["accepted"]
    assert np.isclose(frame.loc[mask, "absolute_error_kg"].mean(), published["accepted_mae_kg"])
    assert np.isclose(frame.loc[mask, "reference_fuel_kg"].sum() /
                      frame.reference_fuel_kg.sum(), published["accepted_reference_fuel_fraction"])

comparison = pd.DataFrame({name: {
    "Accepted (%)": 100 * mask.mean(),
    "Accepted MAE (kg)": frame.loc[mask, "absolute_error_kg"].mean(),
    "Rejected MAE (kg)": frame.loc[~mask, "absolute_error_kg"].mean(),
    "Labeled fuel represented (%)": 100 * frame.loc[mask, "reference_fuel_kg"].sum() /
                                     frame.reference_fuel_kg.sum()
} for name, mask in selected.items()}).T
print("Replayed selections match the published JSON.")
display(comparison.round(2))

## 5. Which groups change?

A low accepted MAE can be achieved by excluding harder records. The next figure plots **coverage** and **accepted-case error** together. The error bars are not confidence intervals; each policy accepts a different set of intervals. Aircraft types with fewer than 50 rows have no guaranteed floor.

In [ ]:
groups = ["<5 min", "5–15 min", "15–60 min"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
colors = {"risk_only": "#b65b32", "representation_floor": "#267c78"}
for name, mask in selected.items():
    access, errors = [], []
    for group in groups:
        members = frame.duration_group.eq(group).to_numpy()
        kept = members & mask
        access.append(100 * kept.sum() / members.sum())
        errors.append(frame.loc[kept, "absolute_error_kg"].mean() if kept.any() else np.nan)
    axes[0].plot(groups, access, marker="o", color=colors[name], label=name)
    axes[1].plot(groups, errors, marker="o", color=colors[name], label=name)
axes[0].set(title="Who receives an estimate?", ylabel="Accepted within group (%)", ylim=(0, 105))
axes[1].set(title="What is the error on accepted cases?", ylabel="Accepted MAE (kg)")
for ax in axes:
    ax.set_xlabel("Interval duration")
    ax.tick_params(axis="x", rotation=12)
    ax.grid(alpha=0.2)
axes[0].legend()
plt.tight_layout()
plt.show()

b744 = frame.aircraft_type.eq("B744").to_numpy()
display(pd.DataFrame({name: {
    "B744 accepted / total": f"{int((mask & b744).sum())} / {int(b744.sum())}",
    "B744 accepted MAE (kg)": frame.loc[mask & b744, "absolute_error_kg"].mean()
} for name, mask in selected.items()}).T.round(2))

**Interpretation:** The 50% floor restores long-interval and B744 access at a real error cost. In September it accepts 33/66 B744 intervals with 609.90 kg accepted MAE. The result is useful for deciding whether an analysis has adequate representation; it is not proof that those B744 fuel estimates are operationally useful.

## 6. Does the error score rank mistakes?

The risk model was fitted on early-September errors, then evaluated on this later batch. Its ten equally sized score groups let us check whether predicted risk and observed absolute error move together. **Do not read the score as a calibrated upper bound**: group means can differ and individual errors can be much larger. A risk–coverage curve also shows the accuracy/acceptance tradeoff across targets; the group floor deliberately changes that tradeoff.

In [ ]:
deciles = pd.DataFrame(results["periods"]["validation"]["residual_diagnostics"]["risk_deciles"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(range(1, len(deciles)+1), deciles.mean_risk_score_kg, marker="o", label="Predicted score")
axes[0].plot(range(1, len(deciles)+1), deciles.observed_mae_kg, marker="o", label="Observed MAE")
axes[0].set(title="Risk-score deciles", xlabel="Low → high score decile", ylabel="Mean error (kg)")
axes[0].legend()
for floor, label, color in [(0, "Risk only", "#b65b32"), (.5, "50% group floor", "#267c78")]:
    points = []
    for target in np.linspace(.5, 1.0, 11):
        mask, _ = select(frame.risk_score_kg.to_numpy(), frame.aircraft_type.to_numpy(),
                         frame.duration_min.to_numpy(), frame.tie.to_numpy(),
                         target=float(target), floor=floor, min_type_n=50)
        points.append((100 * mask.mean(), frame.loc[mask, "absolute_error_kg"].mean()))
    axes[1].plot(*np.array(points).T, marker="o", color=color, label=label)
axes[1].set(title="Error versus actual coverage", xlabel="Intervals accepted (%)", ylabel="Accepted MAE (kg)")
axes[1].legend()
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 7. What does this study establish?

- **Finding:** A lower-error selector can almost eliminate access for some aircraft and longer intervals. A transparent group floor increases access and the share of this *labeled sample's* fuel represented, but also increases accepted error.
- **Not a fresh test:** The access policy was conceived after historical October model results were examined. Current September/October policy comparisons are exploratory. The former Phase 4 model also used different internal early stopping, so its old score cannot isolate a policy effect.
- **Data limits:** Labels come from differences in reported onboard fuel. Aircraft weight, payload, winds and weather are absent; raw altitude/speed units are unverified. Routes recur across periods, and the benchmark supplies completed interval boundaries. No causal or real-time fuel-burn claim follows.
- **Presentation answer:** The beneficiaries are aviation researchers and public-data curators assessing *sample coverage*. The general ML lesson is that selective prediction must be evaluated for both error **and who it excludes**.

Read [method details](../docs/methods.md), [source and limitations](../README.md), and the [pre-submission audit](../docs/pre-submission-audit.md). To recreate the raw features rather than replay committed scores, follow the download and extraction commands in the README.